# The Fingers of the 6L2 component

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from wakis import WakeSolver
from wakis import SolverFIT3D
#from wakis import GridFIT3D 
#from  clara_gridFIT3D_markCellsinSTL_WIP import GridFIT3D
#from wakis import clara_gridFIT3D_markCellsinSTL_WIP 
from wakis.clara_gridFIT3D_markCellsinSTL_WIP import GridFIT3D

import numpy as np
import pyvista as pv

from pyvista.trame.jupyter import launch_server
await launch_server().ready

#pv.set_jupyter_backend('html') 
pv.set_jupyter_backend('trame') 

from benchmark import benchmark

import pickle

ERROR:wslink.protocol:Exception raised
ERROR:wslink.protocol:ClientConnectionResetError('Cannot write to closing transport')
ERROR:wslink.protocol:Traceback (most recent call last):
  File "/home/cwimmelm/miniforge3/envs/wakis-env/lib/python3.11/site-packages/wslink/protocol.py", line 322, in onCompleteMessage
    await self.sendWrappedMessage(
  File "/home/cwimmelm/miniforge3/envs/wakis-env/lib/python3.11/site-packages/wslink/protocol.py", line 426, in sendWrappedMessage
    await ws.send_bytes(chunk)
  File "/home/cwimmelm/miniforge3/envs/wakis-env/lib/python3.11/site-packages/aiohttp/web_ws.py", line 426, in send_bytes
    await self._writer.send_frame(data, WSMsgType.BINARY, compress=compress)
  File "/home/cwimmelm/miniforge3/envs/wakis-env/lib/python3.11/site-packages/aiohttp/_websocket/writer.py", line 90, in send_frame
    self._send_compressed_frame_sync(message, opcode, compress)
  File "/home/cwimmelm/miniforge3/envs/wakis-env/lib/python3.11/site-packages/aiohttp/_websocket

In [4]:

def read_stl( key):
    # import stl
    surf = pv.read(stl_solids[key])
    '''# rotate
    surf = surf.rotate_x(stl_rotate[key][0])
    surf = surf.rotate_y(stl_rotate[key][1])
    surf = surf.rotate_z(stl_rotate[key][2])
    # translate
    surf = surf.translate(stl_translate[key])
    # scale
    surf = surf.scale(stl_scale[key])'''
    return surf


def plot_stl_mask(stl_solid, cmap='viridis', bounding_box=True, show_grid=True,
                      add_stl='all', stl_opacity=0., stl_colors=None,
                      xmax=None, ymax=None, zmax=None,
                      anti_aliasing='ssaa', smooth_shading=False, off_screen=False):

        pv.global_theme.allow_empty_mesh = True
        pl = pv.Plotter()
        vals = {'x':xmax, 'y':ymax, 'z':zmax}

        # --- Update function ---
        def update_clip(val, axis="x"):
            vals[axis] = val
            # define bounds dynamically
            if axis == "x":
                slice_obj = grid.slice(normal="x", origin=(val, 0, 0))
            elif axis == "y":
                slice_obj = grid.slice(normal="y", origin=(0, val, 0))
            else:  # z
                slice_obj = grid.slice(normal="z", origin=(0, 0, val))

            # add clipped volume (scalars)
            pl.add_mesh(
                grid.clip_box(bounds=(xmin, vals['x'],
                                           ymin, vals['y'],
                                           zmin, vals['z']), invert=False),
                scalars=stl_solid,
                cmap=cmap,
                name="clip",
            )

            # add slice wireframe (grid structure)
            if show_grid:
                pl.add_mesh(slice_obj, style="wireframe", color="grey", name="slice")

        # Plot stl surface(s)
        if add_stl is not None:
            if type(add_stl) is str: #add all stl solids
                if add_stl.lower() == 'all':
                    for i, key in enumerate(stl_solids):
                        surf = read_stl(key)
                        if type(stl_colors) is dict:
                            pl.add_mesh(surf, color=stl_colors[key], opacity=stl_opacity, silhouette=dict(color=stl_colors[key]), name=key)
                        elif type(stl_colors) is list:
                            pl.add_mesh(surf, color=stl_colors[i], opacity=stl_opacity, silhouette=dict(color=stl_colors[i]), name=key)
                        else:
                            pl.add_mesh(surf, color='white', opacity=stl_opacity, silhouette=True, name=key)
                else: #add 1 selected stl solid
                    key = add_stl
                    surf = read_stl(key)
                    pl.add_mesh(surf, color=stl_colors[key], opacity=stl_opacity, silhouette=dict(color=stl_colors[key]), name=key)

            elif type(add_stl) is list: #add selected list of stl solids
                for i, key in enumerate(add_stl):
                    surf = read_stl(key)
                    if type(stl_colors[key]) is dict:
                        pl.add_mesh(surf, color=stl_colors[key], opacity=stl_opacity, silhouette=dict(color=stl_colors[key]), name=key)
                    elif type(stl_colors) is list:
                        pl.add_mesh(surf, color=stl_colors[i], opacity=stl_opacity, silhouette=dict(color=stl_colors[i]), name=key)
                    else:
                        pl.add_mesh(surf, color='white', opacity=stl_opacity, silhouette=True, name=key)

        # --- Sliders (placed side-by-side vertically) ---
        pl.add_slider_widget(
            lambda val: update_clip(val, "x"),
            [xmin, xmax],
            value=xmax, title="X Clip",
            pointa=(0.8, 0.8), pointb=(0.95, 0.8),  # top-right
            style='modern',
        )

        pl.add_slider_widget(
            lambda val: update_clip(val, "y"),
            [ymin, ymax],
            value=ymax, title="Y Clip",
            pointa=(0.8, 0.6), pointb=(0.95, 0.6),  # middle-right
            style='modern',
        )

        pl.add_slider_widget(
            lambda val: update_clip(val, "z"),
            [zmin, zmax],
            value=zmax, title="Z Clip",
            pointa=(0.8, 0.4), pointb=(0.95, 0.4),  # lower-right
            style='modern',
        )

        # Camera orientation
        pl.camera_position = 'zx'
        pl.camera.azimuth += 30
        pl.camera.elevation += 30
        pl.set_background('mistyrose', top='white')
        #self._add_logo_widget(pl)
        pl.add_axes()
        pl.enable_3_lights()
        pl.enable_anti_aliasing(anti_aliasing)

        if bounding_box:
            pl.add_mesh(pv.Box(bounds=(xmin, xmax, ymin, ymax, zmin, zmax)),
            style="wireframe", color="black", line_width=2, name="domain_box")

        if off_screen:
            pl.export_html(f'grid_stl_mask_{stl_solid}.html')
        else:
            pl.show()

In [5]:
stl_finger= 'clara_stl_files/012_LHC_WVM_6L2-cube.stl'
surf_finger= pv.read(stl_finger)


materials = {'Fingers': [1e4, 1., 1e4]}
stl_names = {'Fingers': stl_finger}
basename = 'Fingers'
stl_solids = {m: f'{basename}-{stl_names[m]}.stl' for m in materials}


xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)
stl_tol=1e-3

Nx, Ny, Nz = 100, 100, 200
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)

spacing=np.min([x[1]-x[0], y[1]-y[0], z[1]-z[0]])
stl_tolerance = np.min([np.min(np.diff(x)), np.min(np.diff(y)), np.min(np.diff(z))]) *stl_tol


#### Superficial benchmarks on the Fingers

In [ ]:


gfit_imp = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0,
                     stl_method='implicit_distance')

testgrid = gfit_imp.grid
surf = gfit_imp.read_stl('Fingers')
spacing = [gfit_imp.x[1] - gfit_imp.x[0], gfit_imp.y[1] - gfit_imp.y[0], gfit_imp.z[1] - gfit_imp.z[0]]

vol, area, errvol, errarea = benchmark(testgrid, spacing, surf_finger)
print(f"Volume: {vol}, Area: {area}, Volume Error: {errvol}, Area Error: {errarea}")



In [ ]:


gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0,
                     stl_method='implicit_distance_tol')

testgrid = gfit.grid
surf = gfit.read_stl('Fingers')
spacing = [gfit.x[1] - gfit.x[0], gfit.y[1] - gfit.y[0], gfit.z[1] - gfit.z[0]]

vol, area, errvol, errarea = benchmark(testgrid, spacing, surf_finger)
print(f"Volume: {vol}, Area: {area}, Volume Error: {errvol}, Area Error: {errarea}")




In [ ]:

# threshold only works on a mesh object, and the point_data_to_cell_data returns a flat numpy array we must store the cell mask as a mesh/grid (3D)
# as such, I pass the full tesgrid and not the testgrid['Fingers']:
inside = testgrid.threshold(0.5, scalars='Fingers')

pl = pv.Plotter()
pl.add_mesh(surf, style='wireframe', opacity=0.2)
pl.add_mesh(inside, color='red', show_edges=True)
pl.show()

In [ ]:
#inside = testgrid.contour(isosurfaces=[0.1], scalars='Fingers')  #thin shell at the boundary 0.1 distance from the surface.
# #countour needs to be on point data, this switch is very inprecise/ineffecient as it is already done reverse in Wakis
grid_with_points = testgrid.cell_data_to_point_data()
inside = grid_with_points.contour(isosurfaces=[0.1], scalars='Fingers')


pl = pv.Plotter()
pl.add_mesh(surf, style='wireframe', opacity=0.2)
pl.add_mesh(inside, color='red', line_width=2)
pl.show()

In [ ]:
'''
pl = pv.Plotter()
pl.add_mesh(surf, style='wireframe', opacity=0.3, color='blue')
slice_mid = testgrid.slice(normal='z')
pl.add_mesh(slice_mid, scalars='Fingers', cmap='viridis', show_edges=True)
pl.show()'''

# Studying implicit_distance and voxelize_rectilinear for the Fingers (fundamentals)

Wakis' plot stl mask:

In [ ]:
'''grid = pv.RectilinearGrid(x, y, z)
stl_finger= 'clara_stl_files/012_LHC_WVM_6L2-cube.stl'
stl_solids = { 'Fingers': stl_finger}

plot_stl_mask(stl_solid='Fingers', cmap='viridis', bounding_box=True,
              xmax=xmax, ymax=ymax, zmax=zmax,
              show_grid=True, stl_opacity=0.2)'''

### Basic (no transformations, no new methods )

In [7]:
grid = pv.RectilinearGrid(x, y, z)

grid.compute_implicit_distance(surf_finger, inplace=True)
#dist = grid["implicit_distance"] 

grid["Fingers"] = (grid["implicit_distance"] <= stl_tolerance).astype(float)
#inside_voxels = grid.threshold(0.5, scalars="Fingers")

grid_cells = grid.point_data_to_cell_data()
benchmark(grid_cells, spacing, surf_finger,key='Fingers')



(np.float64(42020.92745833598),
 [52133.68830067488, 52133.68830067487, 404741.1368390687],
 np.float64(448.4181387632064),
 [3.731650200323032, 3.731650200323059, 647.3816376624554])

In [ ]:

grid["Fingers"] = (grid["implicit_distance"] <= 0.3).astype(float)
#inside_voxels = grid.threshold(0.5, scalars="Fingers")

grid_cells = grid.point_data_to_cell_data()
benchmark(grid_cells, spacing, surf_finger, key='Fingers')



KeyboardInterrupt: 

: 

In [ ]:


grid["Fingers"] = grid.point_data_to_cell_data()['implicit_distance'] <= 0.3

benchmark(grid, spacing, surf_finger, key='Fingers')



(np.float64(264.5385263720909),
 [2152.4484817873717, 2152.4484817873717, 52319.9390014387],
 np.float64(96.54748871480791),
 [96.02535960672078, 96.02535960672078, 3.387725797580665])

In [12]:
surf_finger.area, surf_finger.volume

(54154.546545317244, 7662.205986312133)

Visualizing the distance calculations to a plane

In [8]:
# Calculate geometric center
center_x = (xmin + xmax) / 2
center_y = (ymin + ymax) / 2
center_z = (zmin + zmax) / 2
diagonal = np.linalg.norm([Lx, Ly, Lz])
plane = pv.Plane(
    center=(center_x, center_y, center_z), 
    direction=(0, 0, 1),  # slicing through the Z axis
    i_size=diagonal, 
    j_size=diagonal, 
    i_resolution=Nx-1, 
    j_resolution=Nz-1
)

plane.compute_implicit_distance(surf_finger, inplace=True)

pl = pv.Plotter()
pl.add_mesh(plane,scalars='implicit_distance', cmap='bwr', clim=[-0.15, 0.7])
pl.add_mesh(surf_finger, color='w', style='wireframe', opacity=0.5)
pl.show()

Widget(value='<iframe src="http://localhost:37049/index.html?ui=P_0x7fd98d4de710_1&reconnect=auto" class="pyvi…

how to choose the right threshold

In [10]:

grid = pv.RectilinearGrid(x, y, z)
grid.compute_implicit_distance(surf_finger, inplace=True)

# OPS; THIS IS HOW TO EXTRACT THE MASK!!!!!!!!!!!
grid['Fingers']= grid.point_data_to_cell_data()['implicit_distance'] <= 0.3  #What is done in Wakis: boolean mask of whole domain

inside_voxels = grid.threshold(0.1, scalars="Fingers")  #we need to use this to obtain the inside mask


pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior mask')
#pl.add_mesh(grid, scalars='implicit_distance',color='red', show_edges=True, label='Grid implicit distances')
#pl.add_mesh(grid, scalars='Fingers',color='red', show_edges=True, label='Grid full domain mask')
pl.add_legend()
pl.show()

key='Fingers'
benchmark(grid, spacing, surf_finger, key)

Widget(value='<iframe src="http://localhost:37049/index.html?ui=P_0x7fd98d4e51d0_3&reconnect=auto" class="pyvi…

(np.float64(264.5385263720909),
 [2152.4484817873717, 2152.4484817873717, 52319.9390014387],
 np.float64(96.54748871480791),
 [96.02535960672078, 96.02535960672078, 3.387725797580665])

#### Computing the absolute distance around the surface (to study if it's the signed part of the compute_implicit_distance that fails ...)


Resulting: it's not. actually, the implicit_distance works better, especially for smaller thresholds apparently. i.e. 0.1

In [13]:
from scipy.spatial import KDTree
tree = KDTree(surf_finger.points)
d, idx = tree.query(plane.points)
plane["Absolute_Distance"] = d

pl = pv.Plotter()
pl.add_mesh(plane,scalars='Absolute_Distance', cmap='bwr', clim=[-3, 10])
pl.add_mesh(surf_finger, color='w', style='wireframe', opacity=0.5)
pl.show()



Widget(value='<iframe src="http://localhost:37049/index.html?ui=P_0x7fd986529f50_6&reconnect=auto" class="pyvi…

In [14]:
grid = pv.RectilinearGrid(x, y, z)

tree = KDTree(surf_finger.points)
d, idx = tree.query(grid.points)
grid["Absolute_Distance"] = (d<= 0.1).astype(float)
inside_voxels = grid.threshold(0.5, scalars="Absolute_Distance")

pl = pv.Plotter()
pl.add_mesh(surf, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior')
pl.add_legend()
pl.show()

NameError: name 'surf' is not defined

Again baseline:

In [ ]:
grid['Fingers']= grid.point_data_to_cell_data()['implicit_distance'] <= 0.3  
inside_voxels = grid.threshold(0.1, scalars="Fingers") 

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior mask')
pl.add_legend()

<vtkmodules.vtkRenderingAnnotation.vtkLegendBoxActor(0x5600f6c1f340) at 0x7fd98659a8c0>

### Transformations on the STL surface

In [35]:
grid = pv.RectilinearGrid(x, y, z)

#thick_finger = surf_finger.warp_by_scalar(factor=0.09) 
surf_finger_1 = surf_finger
surf_finger_1.clean().fill_holes(0.1)


center_x, center_y, center_z = (xmin + xmax) / 2, (ymin + ymax) / 2, (zmin + zmax) / 2
diagonal = np.linalg.norm([Lx, Ly, Lz])
plane = pv.Plane(
    center=(center_x, center_y, center_z), 
    direction=(0, 0, 1),  # slicing through the Z axis
    i_size=diagonal, 
    j_size=diagonal, 
    i_resolution=Nx-1, 
    j_resolution=Nz-1)

plane.compute_implicit_distance(surf_finger_1, inplace=True)

#distance plot for the transformed STL surface: 
pl = pv.Plotter()
pl.add_mesh(plane,scalars='implicit_distance', cmap='bwr', clim=[-0.15, 0.7])
pl.add_mesh(surf_finger_1, color='w', style='wireframe', opacity=0.5)
pl.show()



grid.compute_implicit_distance(surf_finger_1, inplace=True)
#extract the mask with the 0.1 threshold (for comparison w. untransformed STL & the unsigned distance)
grid["Fingers"] = (grid["implicit_distance"] <= 0.1).astype(float)
inside_voxels = grid.threshold(0.1, scalars="Fingers")

pl = pv.Plotter()
pl.add_mesh(surf_finger_1, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior')
pl.add_legend()
pl.show()


Widget(value='<iframe src="http://localhost:46391/index.html?ui=P_0x7f30c2d60990_21&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:46391/index.html?ui=P_0x7f30c2ec5f50_22&reconnect=auto" class="pyv…

### A hybrid approach (voxelize + implicit_distance)

In [ ]:
grid = pv.RectilinearGrid(x, y, z)
#grid = pv.ImageData(dimensions=(Nx, Ny, Nz), spacing=spacing, origin=origin)

# Get the "Robust" Binary Mask (The Raster Approach). This handles the thin surfaces without "leaks"
mask = pv.voxelize_rectilinear(surf_finger, x_res=x, y_res=y, z_res=z)
binary_sign = mask.point_data['voxelized'].astype(float)
binary_sign[binary_sign == 0] = -1  # Convert 0/1 to -1/1

# Get the "Precise" Distance (The Implicit Approach). We calculate distance but IGNORE the built-in sign logic if it's flaky
sdf_raw = grid.compute_implicit_distance(surf_finger)
distances = np.abs(sdf_raw.point_data['implicit_distance'])

# Hybrid Merge: Use the rasterized mask to dictate the "Inside/Outside" and the SDF to provide the "How far"
grid.point_data['hybrid_sdf'] = distances * binary_sign

In [ ]:
grid = pv.RectilinearGrid(x, y, z)
from scipy.ndimage import distance_transform_edt

# Generate the robust binary mask (Handles holes/thin walls)
voxels = pv.voxelize_rectilinear(surf_finger, x_res=x, y_res=y, z_res=z)
mask = voxels.point_data['voxelized'].reshape((Nx, Ny, Nz), order='F')

# Compute Distance via Grid Topology (No "projections" to fail)
# This calculates the distance to the nearest "1" (the surface/interior)
dist_outside = distance_transform_edt(mask == 0) # Distance from outside to surface
dist_inside = distance_transform_edt(mask == 1)  # Distance from inside to surface

# Combine into a Signed Distance Field (however limited by the voxel size as resolution increases, but should be robust to holes/thin walls)
# (Outside is positive, Inside is negative)
spacing_avg = np.mean([Lx/(Nx-1), Ly/(Ny-1), Lz/(Nz-1)])
hybrid_sdf = (dist_outside - dist_inside) * spacing_avg

# 4. Push back to PyVista
grid.point_data['robust_sdf'] = hybrid_sdf.flatten(order='F')

## voxelized

In [ ]:
vox = surf.voxelize_rectilinear(spacing=spacing)
testgrid['shell'] = vox.cell_data['mask'].astype(bool)

In [ ]:
xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]   #not exactly the same as the spacing used to create the grid, but should be close enough for testing

voxelized_grid = surf_shell.voxelize_rectilinear(spacing=spacing)

print(voxelized_grid)
print(voxelized_grid['mask'])
print(voxelized_grid.cell_data['mask'])

cpos = pv.CameraPosition(position=(15, 3, 15), focal_point=(0, 0, 0), viewup=(0, 0, 0))
voxelized_grid.plot(scalars='mask', show_edges=False, cpos=cpos)

array_name = 'mask'
is_inside = voxelized_grid.cell_data[array_name].view(np.bool_)

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', style='wireframe', opacity=0.3,label="Origional STL")
inside_voxels = voxelized_grid.extract_cells(voxelized_grid.cell_data[array_name].view(np.bool_))
pl.add_mesh(inside_voxels, color='red', label="Voxelized Interior")
pl.add_legend()
pl.show()